# Australian Speed Signs -- YOLOv5n Training

Train a 9-class YOLOv5n model for the AI-Powered HUD project.

**Target device:** Luckfox Pico Ultra (RV1106G3, 0.5 TOPS NPU, INT8)

| ID | Class | ID | Class |
|----|-------|----|-------|
| 0 | speed_sign_30 | 5 | speed_sign_80 |
| 1 | speed_sign_40 | 6 | speed_sign_100 |
| 2 | speed_sign_50 | 7 | speed_sign_110 |
| 3 | speed_sign_60 | 8 | speed_camera |
| 4 | speed_sign_70 | | |

**Training resolution:** 640x640 (higher resolution improves small sign detection)

**Runtime:** Change to GPU (Runtime > Change runtime type > T4 GPU)

## 1. Environment Setup

In [ ]:
# Check GPU availability
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

In [ ]:
# Clone airockchip/yolov5 (RKNN-optimized fork, required for --rknpu export)
!git clone https://github.com/airockchip/yolov5.git
%cd yolov5
!pip install -r requirements.txt -q
!pip install roboflow -q

# [Fix] onnxscript is required by ONNX export with PyTorch >= 2.6
!pip install onnxscript -q

# [Fix] Pillow 10+ removed font.getsize() used by YOLOv5 utils/plots.py
# Pin to Pillow 9.x to avoid AttributeError during val/detect visualization
!pip install "Pillow<10" -q

## 2. Load Dataset

**Option A (recommended):** Upload pre-prepared `au_speed_dataset.tar.gz` (116MB).
This was already merged and class-mapped locally from AU Traffic Sign + GTSDB.

**Option B:** Download from Roboflow and merge on Colab (requires API key).

Run ONE of the two cells below.

In [ ]:
# ============================================================
# OPTION A: Upload pre-prepared dataset (RECOMMENDED)
# Upload au_speed_dataset.tar.gz from training/ directory
# ============================================================

import os, yaml, shutil
from pathlib import Path
from collections import defaultdict
from google.colab import files

TARGET_CLASSES = [
    "speed_sign_30",  "speed_sign_40",  "speed_sign_50",
    "speed_sign_60",  "speed_sign_70",  "speed_sign_80",
    "speed_sign_100", "speed_sign_110", "speed_camera",
]

DATASET_ROOT = "/content/au_speed_dataset"

print("Upload au_speed_dataset.tar.gz ...")
uploaded = files.upload()

if uploaded:
    fname = list(uploaded.keys())[0]
    !tar -xzf "{fname}" -C /content/
    !rm -f "{fname}"

    # Verify
    for split in ["train", "valid", "test"]:
        img_dir = f"{DATASET_ROOT}/{split}/images"
        n = len(os.listdir(img_dir)) if os.path.exists(img_dir) else 0
        print(f"  {split}: {n} images")

    # Update paths in data.yaml for Colab
    data_yaml = {
        "train": f"{DATASET_ROOT}/train/images",
        "val": f"{DATASET_ROOT}/valid/images",
        "test": f"{DATASET_ROOT}/test/images",
        "nc": 9,
        "names": TARGET_CLASSES,
    }
    with open(f"{DATASET_ROOT}/data.yaml", "w") as f:
        yaml.dump(data_yaml, f, default_flow_style=False, sort_keys=False)

    print(f"\n  Dataset ready at {DATASET_ROOT}")
    print("  Skip cells 2B-3, go directly to cell 4 (Visualize).")
else:
    print("No file uploaded. Use Option B below instead.")

In [ ]:
# ============================================================
# OPTION B: Download from Roboflow (if you skipped Option A)
# ============================================================

ROBOFLOW_API_KEY = "YOUR_API_KEY"  # <-- Replace this!

from roboflow import Roboflow
rf = Roboflow(api_key=ROBOFLOW_API_KEY)

# Source 1: AU Traffic Sign v6 (62 classes, 4371 images)
print("Downloading Australia Traffic Sign v6...")
try:
    au_project = rf.workspace("elec5308-w8jl5").project("australia-traffic-sign")
    au_dataset = au_project.version(6).download("yolov5", location="/content/downloads/au_signs")
    print(f"Done: {au_dataset.location}")
except Exception as e:
    print(f"Failed: {e}")

# Source 2: GTSDB v3 (46 classes, ~900 images)
print("\nDownloading GTSDB v3...")
try:
    gtsdb_project = rf.workspace("mohamed-traore-2ekkp").project(
        "gtsdb---german-traffic-sign-detection-benchmark")
    gtsdb_dataset = gtsdb_project.version(3).download("yolov5", location="/content/downloads/gtsdb")
    print(f"Done: {gtsdb_dataset.location}")
except Exception as e:
    print(f"Failed: {e}")

In [ ]:
# ============================================================
# Inspect downloaded datasets -- see what classes are available
# ============================================================

def inspect_dataset(dataset_dir, name="Dataset"):
    """Read data.yaml and show available classes."""
    yaml_path = os.path.join(dataset_dir, "data.yaml")
    if not os.path.exists(yaml_path):
        print(f"  [{name}] No data.yaml found at {dataset_dir}")
        return None

    with open(yaml_path, "r") as f:
        data = yaml.safe_load(f)

    names = data.get("names", [])
    if isinstance(names, dict):
        names = [names[k] for k in sorted(names.keys())]

    print(f"\n  [{name}] {len(names)} classes:")
    for i, n in enumerate(names):
        print(f"    {i}: {n}")

    # Count images per split
    for split in ["train", "valid", "test"]:
        img_dir = os.path.join(dataset_dir, split, "images")
        if os.path.exists(img_dir):
            count = len([f for f in os.listdir(img_dir)
                        if f.lower().endswith(('.jpg','.jpeg','.png'))])
            print(f"    {split}: {count} images")

    return names

print("=" * 50)
print("  Downloaded Dataset Inspection")
print("=" * 50)

au_classes = inspect_dataset("/content/downloads/au_signs", "AU Traffic Signs")
gtsdb_classes = inspect_dataset("/content/downloads/gtsdb", "GTSDB")

## 3. Build Class Mapping & Merge Datasets

After inspecting the classes above, we build a mapping from each source dataset's
class indices to our target 9 classes. **Review the output of the previous cell**
and adjust the mapping below if class names differ from expectations.

In [ ]:
# ============================================================
# Auto-build class mapping from source names to target IDs
# ============================================================

# Keywords that map to each target class
# Matching is case-insensitive; checks if ANY keyword appears in the class name
TARGET_KEYWORDS = {
    0: ["30"],
    1: ["40"],
    2: ["50"],
    3: ["60"],
    4: ["70"],
    5: ["80"],
    6: ["100"],
    7: ["110"],
    8: ["camera", "speed cam"],
}

# Speed values we DON'T want (would cause false matches)
EXCLUDE_KEYWORDS = ["120", "130", "20", "10", "90"]


def auto_map_classes(source_names, dataset_name=""):
    """
    Automatically map source class names to our 9 target classes.
    Only matches classes whose names contain speed-related keywords.
    """
    mapping = {}
    if source_names is None:
        return mapping

    for src_idx, src_name in enumerate(source_names):
        name_lower = src_name.lower()

        # Must look like a speed/limit/sign/camera class
        is_speed_related = any(kw in name_lower for kw in
            ["speed", "limit", "km", "camera"])

        if not is_speed_related:
            continue

        # Check exclusions first (e.g., "120 km/h" should not match "20")
        is_excluded = any(excl in name_lower for excl in EXCLUDE_KEYWORDS)

        for target_id, keywords in TARGET_KEYWORDS.items():
            for kw in keywords:
                if kw in name_lower:
                    # For numeric matches, verify it's not an excluded value
                    if kw.isdigit() and is_excluded and kw not in ["camera"]:
                        # Double check: "120" contains "20", but we want exact
                        # Only match if the number appears as a standalone value
                        import re
                        pattern = r'(?<!\d)' + re.escape(kw) + r'(?!\d)'
                        if not re.search(pattern, src_name):
                            continue
                    mapping[src_idx] = target_id
                    print(f"  [{dataset_name}] {src_idx}: '{src_name}' "
                          f"-> {target_id}: '{TARGET_CLASSES[target_id]}'")
                    break
            if src_idx in mapping:
                break

    return mapping


print("=" * 55)
print("  Auto Class Mapping")
print("=" * 55)

au_map = auto_map_classes(au_classes, "AU")
gtsdb_map = auto_map_classes(gtsdb_classes, "GTSDB")

print(f"\n  AU: {len(au_map)} classes mapped")
print(f"  GTSDB: {len(gtsdb_map)} classes mapped")

# Show which target classes have NO source data
mapped_targets = set(au_map.values()) | set(gtsdb_map.values())
missing = [f"{i}:{TARGET_CLASSES[i]}" for i in range(9) if i not in mapped_targets]
if missing:
    print(f"\n  [Warning] No source data for: {', '.join(missing)}")
    print("  These classes will need custom-labeled images.")

In [ ]:
# ============================================================
# Merge datasets: remap class IDs, copy images & labels
# ============================================================

from tqdm.notebook import tqdm

stats = defaultdict(int)  # target_class_id -> annotation count


def merge_yolo_dataset(src_dir, dst_root, class_map, prefix=""):
    """
    Copy images & labels from a YOLO-format dataset, remapping class IDs.
    Only keeps annotations for classes present in class_map.
    """
    total = 0
    img_exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

    for split in ["train", "valid", "test"]:
        src_img_dir = os.path.join(src_dir, split, "images")
        src_lbl_dir = os.path.join(src_dir, split, "labels")
        dst_img_dir = os.path.join(dst_root, split, "images")
        dst_lbl_dir = os.path.join(dst_root, split, "labels")

        if not os.path.exists(src_img_dir):
            continue

        img_files = [f for f in os.listdir(src_img_dir)
                     if Path(f).suffix.lower() in img_exts]

        for img_file in tqdm(img_files, desc=f"  {prefix} {split}", leave=False):
            stem = Path(img_file).stem
            src_label = os.path.join(src_lbl_dir, stem + ".txt")
            if not os.path.exists(src_label):
                continue

            # Read and remap labels
            new_lines = []
            with open(src_label, "r") as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) < 5:
                        continue
                    src_cls = int(parts[0])
                    if src_cls in class_map:
                        target_cls = class_map[src_cls]
                        new_lines.append(f"{target_cls} {' '.join(parts[1:])}")
                        stats[target_cls] += 1

            if not new_lines:
                continue

            # Copy image & write remapped label (prefix avoids name collisions)
            dst_name = f"{prefix}{img_file}"
            shutil.copy2(os.path.join(src_img_dir, img_file),
                         os.path.join(dst_img_dir, dst_name))
            with open(os.path.join(dst_lbl_dir, f"{prefix}{stem}.txt"), "w") as f:
                f.write("\n".join(new_lines) + "\n")
            total += 1

    return total


# Merge AU Traffic Signs
n_au = 0
if au_map:
    print("Merging AU Traffic Sign dataset...")
    n_au = merge_yolo_dataset("/content/downloads/au_signs", DATASET_ROOT,
                              au_map, prefix="au_")
    print(f"  -> {n_au} images merged")

# Merge GTSDB
n_gtsdb = 0
if gtsdb_map:
    print("\nMerging GTSDB dataset...")
    n_gtsdb = merge_yolo_dataset("/content/downloads/gtsdb", DATASET_ROOT,
                                 gtsdb_map, prefix="gtsdb_")
    print(f"  -> {n_gtsdb} images merged")

print(f"\nTotal: {n_au + n_gtsdb} images")

In [ ]:
# ============================================================
# Dataset statistics & create data.yaml
# ============================================================

# Print class distribution
print("=" * 60)
print("  Class Distribution")
print("=" * 60)
total_ann = sum(stats.values())
for cls_id in range(9):
    name = TARGET_CLASSES[cls_id]
    count = stats.get(cls_id, 0)
    pct = (count / total_ann * 100) if total_ann > 0 else 0
    bar = "#" * min(count // 3, 40)
    print(f"  {cls_id}: {name:<20s} {count:>5d} ({pct:>5.1f}%) {bar}")
print(f"  {'TOTAL':<23s} {total_ann:>5d}")
print("=" * 60)

# Count images per split
for split in ["train", "valid", "test"]:
    img_dir = f"{DATASET_ROOT}/{split}/images"
    n = len(os.listdir(img_dir)) if os.path.exists(img_dir) else 0
    print(f"  {split}: {n} images")

# Warn about weak classes
weak = [(i, TARGET_CLASSES[i], stats.get(i, 0))
        for i in range(9) if stats.get(i, 0) < 20]
if weak:
    print(f"\n  [Warning] Underrepresented classes (< 20 annotations):")
    for i, name, count in weak:
        print(f"    - {name}: {count}")
    print("  Consider adding custom labeled data for these classes.")

# Write data.yaml
data_yaml = {
    "train": f"{DATASET_ROOT}/train/images",
    "val": f"{DATASET_ROOT}/valid/images",
    "test": f"{DATASET_ROOT}/test/images",
    "nc": 9,
    "names": TARGET_CLASSES,
}

yaml_path = f"{DATASET_ROOT}/data.yaml"
with open(yaml_path, "w") as f:
    yaml.dump(data_yaml, f, default_flow_style=False, sort_keys=False)
print(f"\n  data.yaml: {yaml_path}")

## 4. Visualize Sample Images

Quick sanity check -- verify bounding boxes and class labels look correct.

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np
import random

COLORS = [
    (255, 80, 80), (80, 255, 80), (80, 80, 255), (255, 255, 80),
    (255, 80, 255), (80, 255, 255), (255, 160, 80), (160, 80, 255),
    (80, 160, 255),
]

def draw_yolo_boxes(img_path, label_path, class_names):
    """Draw YOLO bounding boxes on an image."""
    img = cv2.imread(img_path)
    if img is None:
        return None
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]

    if os.path.exists(label_path):
        with open(label_path, "r") as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 5:
                    continue
                cls_id = int(parts[0])
                cx, cy, bw, bh = [float(x) for x in parts[1:5]]

                x1 = int((cx - bw / 2) * w)
                y1 = int((cy - bh / 2) * h)
                x2 = int((cx + bw / 2) * w)
                y2 = int((cy + bh / 2) * h)

                color = COLORS[cls_id % len(COLORS)]
                cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
                label = f"{class_names[cls_id]}" if cls_id < len(class_names) else f"cls{cls_id}"
                cv2.putText(img, label, (x1, y1 - 5),
                           cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)
    return img


# Show 8 random training samples
train_img_dir = f"{DATASET_ROOT}/train/images"
train_lbl_dir = f"{DATASET_ROOT}/train/labels"
all_imgs = [f for f in os.listdir(train_img_dir)
            if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

if all_imgs:
    samples = random.sample(all_imgs, min(8, len(all_imgs)))

    fig, axes = plt.subplots(2, 4, figsize=(20, 10))
    for ax, img_file in zip(axes.flat, samples):
        stem = Path(img_file).stem
        img = draw_yolo_boxes(
            os.path.join(train_img_dir, img_file),
            os.path.join(train_lbl_dir, stem + ".txt"),
            TARGET_CLASSES,
        )
        if img is not None:
            ax.imshow(img)
            ax.set_title(img_file[:30], fontsize=8)
        ax.axis("off")
    plt.suptitle("Training Samples with Bounding Boxes", fontsize=14)
    plt.tight_layout()
    plt.show()
else:
    print("No training images found. Check dataset preparation above.")

## 5. Train YOLOv5n

Transfer learning from COCO-pretrained weights. Training at 640x640 for better
small sign detection accuracy. The RKNN model input size is queried at runtime
from the model file, so the C code adapts automatically.

Expected training time: ~40-60 min on T4 GPU for 100 epochs.

**Compatibility fixes applied automatically:**
- PyTorch >= 2.6: `torch.load()` default changed to `weights_only=True`, patched to `False`
- Pillow 10+: `font.getsize()` removed, patched to `font.getbbox()` (if Pillow pin failed)

In [ ]:
%cd /content/yolov5

# ============================================================
# [Fix] PyTorch >= 2.6: torch.load() default weights_only=True
# breaks loading YOLOv5 .pt checkpoints.
#
# IMPORTANT: Do NOT use sed for this! sed regex cannot handle
# nested parentheses like torch.load(attempt_download(w), ...)
# which causes weights_only=False to be inserted into the wrong
# function call. Use Python-based patching instead.
# ============================================================
import subprocess

def patch_torch_load(filepath):
    """Add weights_only=False to all torch.load() calls, handling nested parens."""
    with open(filepath, 'r') as f:
        content = f.read()

    original = content
    result = []
    i = 0

    while i < len(content):
        idx = content.find('torch.load(', i)
        if idx == -1:
            result.append(content[i:])
            break

        result.append(content[i:idx])

        # Find matching closing paren (correctly handles nesting)
        start = idx + len('torch.load(')
        depth = 1
        j = start
        while j < len(content) and depth > 0:
            if content[j] == '(':
                depth += 1
            elif content[j] == ')':
                depth -= 1
            j += 1

        args_str = content[start:j-1]

        if 'weights_only' not in args_str:
            result.append(f'torch.load({args_str}, weights_only=False)')
        else:
            result.append(f'torch.load({args_str})')

        i = j

    content = ''.join(result)

    if content != original:
        with open(filepath, 'w') as f:
            f.write(content)
        return True
    return False


files_out = subprocess.run(
    ["grep", "-rl", "torch.load", "."],
    capture_output=True, text=True
).stdout.strip().split("\n")

patched = 0
for f in files_out:
    if f.endswith('.py'):
        if patch_torch_load(f):
            patched += 1
            print(f"  Patched: {f}")
print(f"torch.load patch: {patched} files fixed\n")

# ============================================================
# [Fix] Pillow 10+ removed font.getsize() in YOLOv5 plots.py
# Fallback patch in case "pip install Pillow<10" was overridden
# ============================================================
plots_py = "utils/plots.py"
with open(plots_py, 'r') as f:
    plots_content = f.read()

if 'getsize' in plots_content:
    plots_content = plots_content.replace(
        'w, h = self.font.getsize(label)',
        'try:\n                w, h = self.font.getsize(label)\n            except AttributeError:\n                bbox = self.font.getbbox(label); w, h = bbox[2] - bbox[0], bbox[3] - bbox[1]'
    )
    with open(plots_py, 'w') as f:
        f.write(plots_content)
    print("Pillow getsize patch: utils/plots.py fixed\n")

# ============================================================
# Train YOLOv5n with transfer learning
# ============================================================
!python train.py \
    --data "/content/au_speed_dataset/data.yaml" \
    --cfg yolov5n.yaml \
    --weights yolov5n.pt \
    --img 640 \
    --batch-size 16 \
    --epochs 100 \
    --workers 2 \
    --project runs/au_speed_signs \
    --name v1 \
    --exist-ok \
    --cache ram

## 6. Evaluate Results

In [ ]:
# Show training curves
from IPython.display import Image, display

results_dir = "/content/yolov5/runs/au_speed_signs/v1"

# Training curves
results_img = f"{results_dir}/results.png"
if os.path.exists(results_img):
    display(Image(filename=results_img, width=900))

# Confusion matrix
cm_img = f"{results_dir}/confusion_matrix.png"
if os.path.exists(cm_img):
    print("\nConfusion Matrix:")
    display(Image(filename=cm_img, width=600))

# PR curve
pr_img = f"{results_dir}/PR_curve.png"
if os.path.exists(pr_img):
    print("\nPR Curve:")
    display(Image(filename=pr_img, width=600))

In [ ]:
# Run validation on test set
!python val.py \
    --data "{DATASET_ROOT}/data.yaml" \
    --weights runs/au_speed_signs/v1/weights/best.pt \
    --img 640 \
    --task test \
    --verbose

In [ ]:
# Inference on sample images to visually verify
!python detect.py \
    --weights runs/au_speed_signs/v1/weights/best.pt \
    --img 640 \
    --conf 0.25 \
    --source "{DATASET_ROOT}/test/images" \
    --project runs/au_speed_signs \
    --name test_detect \
    --exist-ok \
    --save-txt \
    --max-det 20

# Show detection results
detect_dir = "/content/yolov5/runs/au_speed_signs/test_detect"
if os.path.exists(detect_dir):
    det_imgs = [f for f in os.listdir(detect_dir)
                if f.lower().endswith(('.jpg', '.jpeg', '.png'))][:8]
    if det_imgs:
        fig, axes = plt.subplots(2, 4, figsize=(20, 10))
        for ax, img_file in zip(axes.flat, det_imgs):
            img = cv2.imread(os.path.join(detect_dir, img_file))
            if img is not None:
                ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
            ax.axis("off")
        plt.suptitle("Detection Results on Test Set", fontsize=14)
        plt.tight_layout()
        plt.show()

## 7. Export ONNX for RKNN

Export with `--rknpu` flag to remove post-processing subgraphs incompatible
with INT8 quantization. This is **required** for the airockchip fork.

Note: Export at 640x640 to match training resolution.

In [ ]:
%cd /content/yolov5

# Export to ONNX with --rknpu flag (required for RKNN conversion)
# [Fix] onnxscript must be installed (done in Environment Setup cell)
!python export.py \
    --weights runs/au_speed_signs/v1/weights/best.pt \
    --img-size 640 640 \
    --batch-size 1 \
    --rknpu \
    --include onnx

# Verify export
onnx_path = "runs/au_speed_signs/v1/weights/best.onnx"
if os.path.exists(onnx_path):
    size_mb = os.path.getsize(onnx_path) / 1024 / 1024
    print(f"\nONNX model exported: {onnx_path}")
    print(f"Size: {size_mb:.2f} MB")
else:
    print("ONNX export failed! Check logs above.")
    print("If 'No module named onnxscript', run: !pip install onnxscript")

## 8. Convert to RKNN (Optional -- requires Linux x86_64)

Convert ONNX to RKNN (INT8 quantized) for deployment on RV1106 NPU.

Colab is Linux x86_64, so we CAN run rknn-toolkit2 here. However, Colab's
pre-installed packages (torch 2.10+, numpy 2.x) conflict with rknn-toolkit2's
requirements (torch<=2.4.0, numpy<=1.26.4).

**Solution:** Use an isolated `virtualenv` with its own compatible dependencies.

Note: Conversion uses 640x640 input to match training resolution.

In [ ]:
%%writefile /content/convert_rknn.py
# ONNX -> RKNN conversion for RV1106 (INT8 quantization)
import glob, os
from rknn.api import RKNN

ONNX_PATH = "/content/yolov5/runs/au_speed_signs/v1/weights/best.onnx"
RKNN_PATH = "/content/au_speed_signs_rv1106.rknn"
DATASET_ROOT = "/content/au_speed_dataset"

# Prepare calibration images (50 representative training images)
cal_images = sorted(glob.glob(f"{DATASET_ROOT}/train/images/*.jpg"))[:50]
if len(cal_images) < 50:
    cal_images += sorted(glob.glob(f"{DATASET_ROOT}/train/images/*.png"))[:50 - len(cal_images)]

cal_file = "/content/dataset.txt"
with open(cal_file, "w") as f:
    f.write("\n".join(cal_images))
print(f"Calibration images: {len(cal_images)}")

# Initialize RKNN
rknn = RKNN(verbose=False)

# YOLOv5 normalization: input 0-255 -> output 0-1
# mean=[0,0,0], std=[255,255,255] => output = (input - 0) / 255
rknn.config(
    mean_values=[[0, 0, 0]],
    std_values=[[255, 255, 255]],
    target_platform="rv1106",
)

print("Loading ONNX...")
ret = rknn.load_onnx(model=ONNX_PATH)
assert ret == 0, f"Load ONNX failed: {ret}"

print("Building RKNN (INT8 quantization)...")
ret = rknn.build(do_quantization=True, dataset=cal_file)
assert ret == 0, f"Build failed: {ret}"

print("Exporting...")
ret = rknn.export_rknn(RKNN_PATH)
assert ret == 0, f"Export failed: {ret}"

rknn.release()

size_mb = os.path.getsize(RKNN_PATH) / 1024 / 1024
print(f"\nDone! {RKNN_PATH}: {size_mb:.2f} MB")
print(f"Platform: rv1106 (INT8 quantized)")

In [ ]:
# ============================================================
# Run conversion in isolated virtualenv
#
# [Why virtualenv?]
# Colab has torch 2.10+ and numpy 2.x, but rknn-toolkit2 requires
# torch<=2.4.0 and numpy<=1.26.4. Direct pip install causes
# ContextualVersionConflict at import time, and monkey-patching
# pkg_resources is fragile. A clean virtualenv is the only reliable
# solution.
#
# [Why not venv?]
# Python's built-in venv fails on Colab because ensurepip is missing.
# virtualenv bundles its own pip and works reliably.
#
# [Known dependency fixes]
# - setuptools<70: newer versions removed pkg_resources module
# - onnx==1.16.2: onnx 1.17+ removed onnx.mapping used by rknn-toolkit2
# ============================================================

# Install virtualenv if not present
!pip install virtualenv -q

# Create clean environment and install rknn-toolkit2 with compatible deps
!virtualenv /content/rknn_venv 2>/dev/null || (rm -rf /content/rknn_venv && virtualenv /content/rknn_venv)
!/content/rknn_venv/bin/pip install "setuptools<70" -q
!/content/rknn_venv/bin/pip install "onnx==1.16.2" -q
!/content/rknn_venv/bin/pip install rknn-toolkit2 -q

# Run conversion
!/content/rknn_venv/bin/python3 /content/convert_rknn.py

## 9. Download Trained Model

Download all artifacts for deployment on Luckfox Pico Ultra.

In [ ]:
# Download trained model files
from google.colab import files
import os

print("Downloading model artifacts...")
print("=" * 50)

# PyTorch weights (for future fine-tuning)
pt_path = "/content/yolov5/runs/au_speed_signs/v1/weights/best.pt"
if os.path.exists(pt_path):
    print(f"  best.pt: {os.path.getsize(pt_path)/1024/1024:.2f} MB")
    files.download(pt_path)

# ONNX model (for RKNN conversion)
onnx_path = "/content/yolov5/runs/au_speed_signs/v1/weights/best.onnx"
if os.path.exists(onnx_path):
    print(f"  best.onnx: {os.path.getsize(onnx_path)/1024/1024:.2f} MB")
    files.download(onnx_path)

# RKNN model (ready for deployment)
rknn_path = "/content/au_speed_signs_rv1106.rknn"
if os.path.exists(rknn_path):
    print(f"  au_speed_signs_rv1106.rknn: {os.path.getsize(rknn_path)/1024/1024:.2f} MB")
    files.download(rknn_path)

print("\n" + "=" * 50)
print("  Deployment Steps:")
print("=" * 50)
print("  1. Copy models to project:")
print("     cp ~/Downloads/best.onnx hub/models/au_speed_signs.onnx")
print("     cp ~/Downloads/best.pt hub/models/au_speed_signs.pt")
print("     cp ~/Downloads/au_speed_signs_rv1106.rknn hub/models/")
print("")
print("  2. Deploy to Luckfox Pico Ultra:")
print("     adb push hub/models/au_speed_signs_rv1106.rknn /root/model/")
print("")
print("  3. Build ai-hud (default AU config):")
print("     cmake ..")
print("")
print("  4. Run:")
print("     adb shell '/root/ai-hud'")